In [ ]:
# !pip install --upgrade datasets

In [1]:
from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset")
data = dataset["train"]

README.md:   0%|          | 0.00/11.7k [00:00<?, ?B/s]

bitext-retail-banking-llm-chatbot-traini(…): reconstructing file:   0%|          |  0.00B / 7.87MB            

bitext-retail-banking-llm-chatbot-traini(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25545 [00:00<?, ? examples/s]

In [2]:

def format_example(example):
    return {
        "text": f"User: {example['instruction']}\nBot: {example['response']}"
    }

data = dataset["train"].map(
    format_example,
    remove_columns=dataset["train"].column_names
)

print(data[0])

Map:   0%|          | 0/25545 [00:00<?, ? examples/s]

{'text': 'User: I would like to acivate a card, can you help me?\nBot: I\'m here to assist you with that! Activating your card is an important step to starting and enjoying its benefits. Here\'s how you can activate your card:\n\n1. Locate the activation instructions: Depending on the card issuer, you may find the activation instructions on a sticker attached to the card itself, in the welcome package, or on the issuer\'s website.\n\n2. Visit the card issuer\'s activation website: Using your computer or mobile device, open a web browser and navigate to the card issuer\'s website. Look for the activation page or section.\n\n3. Enter the required information: Follow the prompts on the activation page and provide the necessary information. This may include your card number, personal details, and security code.\n\n4. Set up your card: Once you\'ve entered the required information, you may have the option to set up a PIN, create an online account, or choose additional security features. Fol

In [3]:
from transformers import AutoTokenizer

model_name = "gpt2"  # نفس الـ tokenizer اللي الموديل بتاعك اتدرب بيه

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

data = data.map(tokenize)
data.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/25545 [00:00<?, ? examples/s]

In [5]:
split_data = data.train_test_split(test_size=0.1, seed=42)
train_dataset = split_data["train"]
eval_dataset = split_data["test"]

In [6]:
import os
import torch
from transformers import GPT2Config, GPT2LMHeadModel

from google.colab import drive
drive.mount("/content/drive")

MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/gpt2_124M_the_verdict_continued_pretraining.pth"
assert os.path.exists(MODEL_PATH), "الملف مش موجود، راجع المسار"


def load_my_gpt2(path=MODEL_PATH):
    """بيحمّل أوزانك (GPTModel بتاع الكتاب) جوه GPT2LMHeadModel بتاع HuggingFace"""
    ckpt = torch.load(path, map_location="cpu")
    sd = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt

    cfg = GPT2Config(vocab_size=50257, n_positions=1024, n_embd=768,
                     n_layer=12, n_head=12, tie_word_embeddings=False)
    m = GPT2LMHeadModel(cfg)

    new = {
        "transformer.wte.weight": sd["tok_emb.weight"],
        "transformer.wpe.weight": sd["pos_emb.weight"],
        "transformer.ln_f.weight": sd["final_norm.scale"],
        "transformer.ln_f.bias": sd["final_norm.shift"],
        "lm_head.weight": sd["out_head.weight"],
    }
    for i in range(cfg.n_layer):
        p, h = f"trf_blocks.{i}.", f"transformer.h.{i}."
        q, k, v = (sd[p + f"att.W_{n}.weight"] for n in ("query", "key", "value"))
        new[h + "attn.c_attn.weight"] = torch.cat([q, k, v], dim=0).T.contiguous()
        if p + "att.W_query.bias" in sd:
            new[h + "attn.c_attn.bias"] = torch.cat([sd[p + f"att.W_{n}.bias"] for n in ("query", "key", "value")])
        else:
            new[h + "attn.c_attn.bias"] = torch.zeros(3 * cfg.n_embd)
        new[h + "attn.c_proj.weight"] = sd[p + "att.out_proj.weight"].T.contiguous()
        new[h + "attn.c_proj.bias"] = sd[p + "att.out_proj.bias"]
        new[h + "ln_1.weight"], new[h + "ln_1.bias"] = sd[p + "norm1.scale"], sd[p + "norm1.shift"]
        new[h + "ln_2.weight"], new[h + "ln_2.bias"] = sd[p + "norm2.scale"], sd[p + "norm2.shift"]
        new[h + "mlp.c_fc.weight"] = sd[p + "ff.layers.0.weight"].T.contiguous()
        new[h + "mlp.c_fc.bias"] = sd[p + "ff.layers.0.bias"]
        new[h + "mlp.c_proj.weight"] = sd[p + "ff.layers.2.weight"].T.contiguous()
        new[h + "mlp.c_proj.bias"] = sd[p + "ff.layers.2.bias"]

    missing, unexpected = m.load_state_dict(new, strict=False)
    missing = [k for k in missing if not k.endswith(("attn.bias", "attn.masked_bias"))]
    assert not missing and not unexpected, (missing, unexpected)
    return m

print("Loading my model...")
model = load_my_gpt2()


Mounted at /content/drive
Loading my model...


In [9]:
# !pip install torchao --upgrade

In [10]:

from peft import LoraConfig, get_peft_model, TaskType

print("5. Applying LoRA adapters...")
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj", "c_fc"],
    fan_in_fan_out=True
)

model.enable_input_require_grads()
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

5. Applying LoRA adapters...
trainable params: 2,359,296 || all params: 165,396,480 || trainable%: 1.4264


In [11]:
from transformers import Trainer, TrainingArguments
print("6. Setting up Training Arguments...")
training_args = TrainingArguments(
    output_dir="./qwen_banking_chatbot",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    fp16=True,
    bf16=False,
    local_rank=-1,
    report_to="none"
)

6. Setting up Training Arguments...


In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer
)
print("Starting training with Trainer...")
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting training with Trainer...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.950295,0.841367


TrainOutput(global_step=2874, training_loss=1.0968657812478233, metrics={'train_runtime': 3043.7055, 'train_samples_per_second': 7.553, 'train_steps_per_second': 0.944, 'total_flos': 2224920060887040.0, 'train_loss': 1.0968657812478233, 'epoch': 1.0})

In [17]:
  import torch


  model.eval()
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model.to(device)

  def chat_with_bot(user_question):

      prompt = f"User: {user_question}\nBot:"
      inputs = tokenizer(prompt, return_tensors="pt").to(device)

      with torch.no_grad():
          output_ids = model.generate(
              **inputs,
              max_new_tokens=1000,
              pad_token_id=tokenizer.eos_token_id,
              no_repeat_ngram_size=3,
              do_sample=True,
              repetition_penalty=1.2,
              temperature=0.4,
              top_k=40,
              top_p=0.85
          )

      full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
      bot_response = full_text.split("Bot:")[-1].strip()
      return bot_response

  # =========================================================
  # 2. حلقة المحادثة الحية (Interactive Chat Loop)
  # =========================================================
  print("--- 🤖 البوت المصرفي جاهز الآن للمحادثة الحية ---")
  print("اكتب سؤالك بالإنجليزية، وللخروج تماماً وإغلاق الشات اكتب: exit\n")

  while True:
      user_input = input("You: ")
      if user_input.strip().lower() == 'exit':
          print("Bot: Thank you for using our banking service. Goodbye! 👋")
          break
      if user_input.strip() == "":
          continue

      response = chat_with_bot(user_input)
      print(f"Bot: {response}\n")

--- 🤖 البوت المصرفي جاهز الآن للمحادثة الحية ---
اكتب سؤالك بالإنجليزية، وللخروج تماماً وإغلاق الشات اكتب: exit

You: I would like to acivate a card, can you help me?
Bot: I'm here for helping you with activating your {{Credit Card}}. Activating your card is an important step to ensure the security of your account and ensure the convenience in using it during your visit abroad or during your stay at home. Here's what you need to do:
1. Contact our customer support team as soonas possible. They are available {{Customer Support Working Hours}} to assist you with the activation process.
2. Provide them with your card details, such as the card number, expiration date (e.g., July 31st), and any other relevant information they may require. This will allow us access into your account securely.
3. They will guide you through the necessary steps and provide you with additional instructions on how long you'll be able activate your card.
4. If you have any further questions about the activation, 

In [15]:
!pip install -q gradio

In [18]:

import gradio as gr
import torch

# التأكد إن الموديل جاهز للاستجابة
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def respond(message, history):
    if not message.strip():
        return "Please enter a message."

    response = chat_with_bot(message)
    return response

# تصميم واجهة المحادثة
demo = gr.ChatInterface(
    fn=respond,
    title="🤖 My Fine-Tuned LLM",
    description=(
        "Chat with my custom GPT-2 model, "
        "fine-tuned on a banking customer-support dataset using LoRA."
    ),
    textbox=gr.Textbox(
        placeholder="Ask your question here...",
        container=True,
        scale=7
    ),
    examples=[
        "How can I check my bank balance?",
        "What should I do if my card is lost?",
        "How can I reset my banking password?"
    ],
    submit_btn="Send"
)

# إنشاء رابط عام مؤقت
demo.launch(share=True, debug=True)


/tmp/ipykernel_1375/2528487354.py:17: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  demo = gr.ChatInterface(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5583727594c253889e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c712178e7b915ea631.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://5583727594c253889e.gradio.live
